# Notebook 6 - STD MAE Plots
This notebook visualizes metrics generated in `5-STD_MAE_Calculations.ipynb`.
Plots are saved under `RESULTS/MAE_TEST/PLOTS`.


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import Parallel, delayed

from src.clinical_combat.robust.robust_utils import (
    add_nb_patients_and_diseased,
    get_metrics,
)

MAIN_FOLDER = 'RESULTS/MAE_TEST'
INIT_HARMONIZATION = 'pairwise'
HARMONIZATION_METHOD = 'pairwise'

PLOTS_FOLDER = f"{MAIN_FOLDER}/PLOTS/{INIT_HARMONIZATION}/{HARMONIZATION_METHOD}"
MAE_PLOT_FOLDER = f"{PLOTS_FOLDER}/MAE_PLOTS"

ROBUST_METHODS = [
    'HC',
    'MLP_EXAMPLE',
    'G_MAD',
    'G_ZS',
    'raw',
    'IQR',
    'SN',
    'QN',
    'MAD',
    'ZS',
    'VS',
    'MMS',
    'FLIP',
    'NO',
]

BASE_COLOR_PALETTES = {
    'HC': 'chartreuse',
    'NO_FILTERING': 'red',
    'MLP': 'yellow',
}
TICK_FONTSIZE = 20
SHOW_ERROR_BARS = False
COLOR_PALETTE = 'gray'

metrics = get_metrics()

# Set to None to auto-detect available values.
diseases = None
sample_sizes = None
disease_ratios = None
num_tests = None
N_JOBS = -1


In [ ]:
def get_ordered_cols(mae_cols, methods=ROBUST_METHODS):
    """Order columns following ROBUST_METHODS and expected renames."""
    rename_map = {'NO': 'NO_FILTERING', 'MLP_EXAMPLE': 'MLP'}
    ordered = []
    mae_set = set(mae_cols)
    for method in methods:
        renamed = rename_map.get(method, method)
        if renamed in mae_set:
            ordered.append(renamed)
        elif method in mae_set:
            ordered.append(method)
    return ordered


def build_color_map(ordered_cols):
    """Return a consistent color map for the provided method columns."""
    col_colors = dict(BASE_COLOR_PALETTES)
    remaining = [col for col in ordered_cols if col not in col_colors]
    if remaining:
        palette = sns.color_palette(COLOR_PALETTE, len(remaining))
        col_colors.update(dict(zip(remaining, palette)))
    return col_colors


def format_ratio(ratio):
    """Format a ratio whether stored as 0-1 or 0-100."""
    value = ratio * 100 if ratio <= 1 else ratio
    return f"{int(round(value))}%"


In [ ]:
def infer_results_grid(mainfolder, harmonization_method, diseases=None):
    """Infer results grid from outputs produced by notebook 4."""
    base_dir = (
        Path(mainfolder)
        / 'PROCESS'
        / INIT_HARMONIZATION
        / harmonization_method
    )
    if not base_dir.exists():
        raise FileNotFoundError(f"{base_dir} not found.")

    disease_candidates = diseases or [
        path.name for path in base_dir.iterdir() if path.is_dir()
    ]

    detected_diseases = []
    sample_sizes = set()
    disease_ratios = set()
    inferred_num_tests = 0

    for disease in sorted(disease_candidates):
        disease_dir = base_dir / disease
        if not disease_dir.is_dir():
            continue
        detected_diseases.append(disease)
        for size_ratio_dir in disease_dir.iterdir():
            if not size_ratio_dir.is_dir() or '_' not in size_ratio_dir.name:
                continue
            size_part, ratio_part = size_ratio_dir.name.split('_', 1)
            try:
                sample_size_val = int(size_part)
                disease_ratio_val = int(ratio_part) / 100
            except ValueError:
                continue

            sample_sizes.add(sample_size_val)
            disease_ratios.add(disease_ratio_val)

            run_ids = [
                int(run_dir.name)
                for run_dir in size_ratio_dir.iterdir()
                if run_dir.is_dir() and run_dir.name.isdigit()
            ]
            if run_ids:
                inferred_num_tests = max(
                    inferred_num_tests, max(run_ids) + 1, len(run_ids)
                )

    detected_diseases = sorted(set(detected_diseases))
    if (
        not detected_diseases
        or not sample_sizes
        or not disease_ratios
        or inferred_num_tests == 0
    ):
        raise ValueError(f"Could not infer parameters from {base_dir}.")
    return (
        detected_diseases,
        sorted(sample_sizes),
        sorted(disease_ratios),
        inferred_num_tests,
    )


In [ ]:
def load_mae_or_maev_compilations(
    mainfolder,
    diseases,
    sample_sizes,
    disease_ratios,
    num_tests,
    mae_or_maev='mae',
):
    """Load concatenated train/test compilations for the provided grid."""
    tests, trains = [], []
    for disease in diseases:
        for sample_size in sample_sizes:
            for ratio in disease_ratios:
                for run_idx in range(num_tests):
                    base = os.path.join(
                        mainfolder,
                        'PROCESS',
                        INIT_HARMONIZATION,
                        HARMONIZATION_METHOD,
                        disease,
                        f"{sample_size}_{int(ratio * 100)}",
                        str(run_idx),
                    )
                    test_path = os.path.join(
                        base, f"{mae_or_maev}_compilation_test.csv"
                    )
                    train_path = os.path.join(
                        base, f"{mae_or_maev}_compilation_train.csv"
                    )
                    if os.path.isfile(test_path):
                        tests.append(pd.read_csv(test_path))
                    if os.path.isfile(train_path):
                        trains.append(pd.read_csv(train_path))
    df_test = pd.concat(tests, ignore_index=True) if tests else pd.DataFrame()
    df_train = pd.concat(trains, ignore_index=True) if trains else pd.DataFrame()
    return df_test, df_train


In [ ]:
def load_compilation(
    mae_or_maev: str,
    split: str,
    *,
    mainfolder: str,
    diseases: list[str],
    sample_sizes: list[int],
    disease_ratios: list[int],
    num_tests: int,
) -> pd.DataFrame:
    if mae_or_maev not in {'mae', 'maev', 'smape', 'std_mae'}:
        raise ValueError("mae_or_maev must be one of: mae, maev, smape, std_mae.")
    if split not in {'test', 'train'}:
        raise ValueError("split must be either 'test' or 'train'.")

    df_test, df_train = load_mae_or_maev_compilations(
        mainfolder,
        diseases,
        sample_sizes,
        disease_ratios,
        num_tests,
        mae_or_maev=mae_or_maev,
    )
    return df_test if split == 'test' else df_train


In [ ]:
(
    detected_diseases,
    detected_sample_sizes,
    detected_disease_ratios,
    detected_num_tests,
) = infer_results_grid(
    MAIN_FOLDER,
    HARMONIZATION_METHOD,
    diseases if diseases is not None else None,
)

if diseases is None:
    diseases = detected_diseases
else:
    missing = sorted(set(diseases) - set(detected_diseases))
    if missing:
        print(f"No data found for: {missing}")
    diseases = [d for d in diseases if d in detected_diseases]
    if not diseases:
        raise ValueError('No valid diseases found in the results.')

sample_sizes = detected_sample_sizes if sample_sizes is None else sample_sizes
disease_ratios = detected_disease_ratios if disease_ratios is None else disease_ratios
num_tests = detected_num_tests if num_tests is None else num_tests

print(f"Diseases: {diseases}")
print(f"Sample sizes: {sample_sizes}")
print(f"Disease ratios: {disease_ratios}")
print(f"Repetitions: {num_tests}")

mae_compilation_train_all = load_compilation(
    'mae',
    'train',
    mainfolder=MAIN_FOLDER,
    diseases=diseases,
    sample_sizes=sample_sizes,
    disease_ratios=disease_ratios,
    num_tests=num_tests,
)
mae_compilation_test_all = load_compilation(
    'mae',
    'test',
    mainfolder=MAIN_FOLDER,
    diseases=diseases,
    sample_sizes=sample_sizes,
    disease_ratios=disease_ratios,
    num_tests=num_tests,
)

smape_compilation_train_all = load_compilation(
    'smape',
    'train',
    mainfolder=MAIN_FOLDER,
    diseases=diseases,
    sample_sizes=sample_sizes,
    disease_ratios=disease_ratios,
    num_tests=num_tests,
)
smape_compilation_test_all = load_compilation(
    'smape',
    'test',
    mainfolder=MAIN_FOLDER,
    diseases=diseases,
    sample_sizes=sample_sizes,
    disease_ratios=disease_ratios,
    num_tests=num_tests,
)

std_mae_compilation_train_all = load_compilation(
    'std_mae',
    'train',
    mainfolder=MAIN_FOLDER,
    diseases=diseases,
    sample_sizes=sample_sizes,
    disease_ratios=disease_ratios,
    num_tests=num_tests,
)
std_mae_compilation_test_all = load_compilation(
    'std_mae',
    'test',
    mainfolder=MAIN_FOLDER,
    diseases=diseases,
    sample_sizes=sample_sizes,
    disease_ratios=disease_ratios,
    num_tests=num_tests,
)


In [ ]:
def wide_to_long(df_large: pd.DataFrame) -> pd.DataFrame:
    """Convert wide MAE compilations to long form."""
    context_cols = ['site', 'robust_method', 'disease', 'metric']
    bundle_cols = [col for col in df_large.columns if col not in context_cols]
    return df_large.melt(
        id_vars=context_cols,
        value_vars=bundle_cols,
        var_name='bundle',
        value_name='mae',
    )


In [ ]:
def plot_mae_mean_all_ratios(
    pivot_df,
    sample_size,
    directory,
    dataset_type,
    y_label='MAE',
):
    """Average metric for each method across diseases, metrics, and ratios."""
    df_filt = pivot_df.loc[
        pivot_df.index.get_level_values('num_patients') == sample_size
    ].reset_index()
    mae_cols = [
        col
        for col in df_filt.columns
        if col
        not in [
            'site',
            'disease',
            'metric',
            'bundle',
            'num_patients',
            'disease_ratio',
            'num_diseased',
        ]
    ]
    ordered_cols = get_ordered_cols(mae_cols)
    if not ordered_cols:
        return

    col_colors = build_color_map(ordered_cols)
    means = [df_filt[col].dropna().mean() for col in ordered_cols]

    x = np.arange(len(ordered_cols))
    bar_width = 0.7

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)
    ax.bar(
        x,
        means,
        width=bar_width,
        color=[col_colors[col] for col in ordered_cols],
        edgecolor='black',
    )

    ax.set_xticks(x)
    ax.set_xticklabels(ordered_cols, rotation=45, ha='right')
    ax.set_title(
        f"{y_label}\nPatients: {sample_size} | Dataset: {dataset_type}"
    )
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    plt.tight_layout()

    out_dir = os.path.join(
        directory,
        f"{y_label}_PLOTS_MEAN",
        'ALL_DISEASES_ALL_METRICS_ALL_RATIOS',
        str(sample_size),
    )
    os.makedirs(out_dir, exist_ok=True)
    fname = f"{y_label}_mean_all_ratios_{dataset_type}.png"
    plt.savefig(os.path.join(out_dir, fname), bbox_inches='tight')
    plt.close()


In [ ]:
def plot_mae_mean_all_diseases_metrics(
    pivot_df,
    sample_size,
    directory,
    dataset_type,
    y_label='MAE',
):
    """Average metric for each method across diseases and metrics per ratio."""
    df_filt = pivot_df.loc[
        pivot_df.index.get_level_values('num_patients') == sample_size
    ].reset_index()

    mae_cols = [
        col
        for col in df_filt.columns
        if col
        not in [
            'site',
            'disease',
            'metric',
            'bundle',
            'num_patients',
            'disease_ratio',
            'num_diseased',
        ]
    ]

    ordered_cols = get_ordered_cols(mae_cols)
    if not ordered_cols:
        return

    col_colors = build_color_map(ordered_cols)

    ratios = sorted(df_filt['disease_ratio'].unique())
    x = np.arange(len(ratios))
    group_width = 0.8
    bar_width = group_width / len(ordered_cols)

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)

    for idx, col in enumerate(ordered_cols):
        means = [
            df_filt[df_filt['disease_ratio'] == ratio][col].dropna().mean()
            for ratio in ratios
        ]
        pos = x - group_width / 2 + (idx + 0.5) * bar_width
        ax.bar(
            pos,
            means,
            width=bar_width * 0.9,
            color=col_colors[col],
            edgecolor='black',
            label=col,
        )

    ax.set_title(
        f"{y_label}\nPatients: {sample_size} | Dataset: {dataset_type}"
    )
    ax.set_xticks(x)
    ax.set_xticklabels([format_ratio(ratio) for ratio in ratios])
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=24)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    plt.tight_layout()

    out_dir = os.path.join(
        directory,
        f"{y_label}_PLOTS_MEAN",
        'ALL_DISEASES_ALL_METRICS',
        str(sample_size),
    )
    os.makedirs(out_dir, exist_ok=True)
    fname = f"{y_label}_mean_all_diseases_metrics_{dataset_type}.png"
    plt.savefig(os.path.join(out_dir, fname), bbox_inches='tight')
    plt.close()


In [ ]:
def _plot_rank_barplot(
    pivot_df,
    sample_size,
    disease,
    directory,
    dataset_type,
    y_label='MAE',
    metric=None,
    aggregate_metrics=False,
):
    """Helper to plot mean metric per method for a given disease/metric."""
    cond = (
        (pivot_df.index.get_level_values('num_patients') == sample_size)
        & (pivot_df.index.get_level_values('disease') == disease)
    )

    if metric is not None:
        cond &= pivot_df.index.get_level_values('metric') == metric

    df_filt = pivot_df.loc[cond].reset_index()
    if df_filt.empty:
        return

    mae_cols = [
        col
        for col in df_filt.columns
        if col
        not in [
            'site',
            'disease',
            'metric',
            'bundle',
            'num_patients',
            'disease_ratio',
            'num_diseased',
        ]
    ]

    ordered_cols = get_ordered_cols(mae_cols)
    if not ordered_cols:
        return

    col_colors = build_color_map(ordered_cols)

    ratios = sorted(df_filt['disease_ratio'].unique())
    x = np.arange(len(ratios))
    group_width = 0.8
    bar_width = group_width / len(ordered_cols)

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)

    for idx, col in enumerate(ordered_cols):
        values_per_ratio = [
            df_filt[df_filt['disease_ratio'] == ratio][col].dropna().values
            for ratio in ratios
        ]
        means = np.array(
            [vals.mean() if len(vals) else np.nan for vals in values_per_ratio],
            dtype=float,
        )
        stds = np.array(
            [vals.std(ddof=0) if len(vals) else 0.0 for vals in values_per_ratio],
            dtype=float,
        )

        pos = x - group_width / 2 + (idx + 0.5) * bar_width
        ax.bar(
            pos,
            means,
            width=bar_width * 0.9,
            color=col_colors[col],
            edgecolor='black',
            label=col,
        )
        if SHOW_ERROR_BARS:
            ax.errorbar(
                pos,
                means,
                yerr=stds,
                fmt='none',
                ecolor='black',
                elinewidth=1,
                capsize=3,
            )

    ax.set_xticks(x)
    ax.set_xticklabels([format_ratio(ratio) for ratio in ratios])

    if aggregate_metrics:
        fname = f"{y_label}_all_metrics_mean_{dataset_type}.png"
    else:
        fname = f"{y_label}_{metric}_mean_{dataset_type}.png"

    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    plt.tight_layout()
    out_dir = os.path.join(
        directory,
        f"{y_label}_PLOTS_MEAN",
        disease,
        str(sample_size),
    )
    os.makedirs(out_dir, exist_ok=True)
    plt.savefig(os.path.join(out_dir, fname), bbox_inches='tight')
    plt.close()


def plot_rank(
    pivot_df,
    sample_size,
    disease,
    metric,
    directory,
    dataset_type,
    y_label='MAE',
):
    """Plot the mean metric for a disease/metric combination."""
    _plot_rank_barplot(
        pivot_df,
        sample_size,
        disease,
        directory,
        dataset_type,
        y_label=y_label,
        metric=metric,
        aggregate_metrics=False,
    )


In [ ]:
def plot_rank_all_metrics(
    pivot_df,
    sample_size,
    disease,
    directory,
    dataset_type,
    y_label='MAE',
):
    """Plot the mean metric aggregated across all metrics."""
    _plot_rank_barplot(
        pivot_df,
        sample_size,
        disease,
        directory,
        dataset_type,
        y_label=y_label,
        metric=None,
        aggregate_metrics=True,
    )


In [ ]:
def plot_mae_all_bundles_pivot(
    pivot_df,
    sample_size,
    disease,
    metric,
    directory,
    dataset_type,
    y_label='MAE',
    bundle=None,
):
    """Boxplots per bundle and ratio for a given disease/metric."""
    cond = (
        (pivot_df.index.get_level_values('num_patients') == sample_size)
        & (pivot_df.index.get_level_values('disease') == disease)
        & (pivot_df.index.get_level_values('metric') == metric)
    )

    if bundle is not None:
        if isinstance(bundle, (list, tuple, set)):
            cond &= pivot_df.index.get_level_values('bundle').isin(bundle)
        else:
            cond &= pivot_df.index.get_level_values('bundle') == bundle

    df_filt = pivot_df.loc[cond].reset_index()
    if df_filt.empty:
        return

    mae_cols = [
        col
        for col in df_filt.columns
        if col
        not in [
            'site',
            'disease',
            'metric',
            'bundle',
            'num_patients',
            'disease_ratio',
            'num_diseased',
        ]
    ]

    ordered_cols = get_ordered_cols(mae_cols)
    if not ordered_cols:
        return

    col_colors = build_color_map(ordered_cols)

    ratios = sorted(df_filt['disease_ratio'].unique())
    x = np.arange(len(ratios))
    group_width = 0.8
    box_width = group_width / len(ordered_cols)

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)

    for idx, col in enumerate(ordered_cols):
        data = [
            df_filt[df_filt['disease_ratio'] == ratio][col].dropna().values
            for ratio in ratios
        ]
        if not any(len(values) for values in data):
            continue

        pos = x - group_width / 2 + (idx + 0.5) * box_width
        ax.boxplot(
            data,
            positions=pos,
            widths=box_width * 0.8,
            patch_artist=True,
            showfliers=False,
            boxprops={
                'facecolor': col_colors[col],
                'edgecolor': col_colors[col],
            },
            medianprops={'color': 'black'},
        )

    bundle_label = (
        ', '.join(bundle) if isinstance(bundle, (list, tuple, set)) else bundle
    )
    title = f"{metric.upper()} evaluation"
    if bundle_label:
        title = f"{title} for {bundle_label.upper()} bundle"
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels([format_ratio(ratio) for ratio in ratios])

    handles = [
        plt.Line2D([0], [0], color=col_colors[col], lw=3, label=col)
        for col in ordered_cols
    ]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1, 1))
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    plt.tight_layout()

    out_dir = os.path.join(directory, f"{y_label}_PLOTS_CANDLES", disease, str(sample_size))
    if bundle is not None:
        out_dir = os.path.join(out_dir, metric)
    os.makedirs(out_dir, exist_ok=True)
    bundle_suffix = bundle_label.replace(' ', '_') if bundle_label else 'all_bundles'
    plt.savefig(
        os.path.join(
            out_dir,
            f"{y_label}_{metric}_{bundle_suffix}_boxplot_{dataset_type}.png",
        ),
        bbox_inches='tight',
    )
    plt.close()


In [ ]:
def plot_mae_each_bundle(
    pivot_df,
    sample_size,
    disease,
    metric,
    directory,
    dataset_type,
    y_label='MAE',
):
    """Plot boxplots for every bundle available for the given configuration."""
    bundles = pivot_df.loc[
        (pivot_df.index.get_level_values('num_patients') == sample_size)
        & (pivot_df.index.get_level_values('disease') == disease)
        & (pivot_df.index.get_level_values('metric') == metric)
    ].index.get_level_values('bundle').unique()

    for bundle in bundles:
        plot_mae_all_bundles_pivot(
            pivot_df,
            sample_size,
            disease,
            metric,
            directory,
            dataset_type,
            y_label=y_label,
            bundle=bundle,
        )


In [ ]:
def plot_mean_vs_sample_size_fixed_ratio(
    pivot_df,
    disease_ratio,
    directory,
    dataset_type,
    y_label='STD_MAE',
    disease='ALL',
    metric=None,
    aggregate_metrics=False,
):
    """Plot mean metric per method vs sample size for a fixed ratio."""

    cond = (
        (pivot_df.index.get_level_values('disease') == disease)
        & (pivot_df.index.get_level_values('disease_ratio') == disease_ratio)
    )
    if (not aggregate_metrics) and (metric is not None):
        cond &= pivot_df.index.get_level_values('metric') == metric

    df_filt = pivot_df.loc[cond].reset_index()
    if df_filt.empty:
        return

    method_cols = [
        col
        for col in df_filt.columns
        if col
        not in [
            'site',
            'disease',
            'metric',
            'bundle',
            'num_patients',
            'disease_ratio',
            'num_diseased',
        ]
    ]

    ordered_cols = get_ordered_cols(method_cols)
    if not ordered_cols:
        return

    col_colors = build_color_map(ordered_cols)

    sizes = sorted(df_filt['num_patients'].unique())
    x = np.arange(len(sizes))
    group_width = 0.8
    bar_width = group_width / len(ordered_cols)
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.set_ylim(0, 0.8)
    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)

    for idx, col in enumerate(ordered_cols):
        means = [
            df_filt[df_filt['num_patients'] == size][col].dropna().mean()
            for size in sizes
        ]

        pos = x - group_width / 2 + (idx + 0.5) * bar_width
        ax.bar(
            pos,
            means,
            width=bar_width * 0.9,
            color=col_colors[col],
            edgecolor='black',
            label=col,
        )

    ratio_value = int(round(disease_ratio * 100)) if disease_ratio <= 1 else int(disease_ratio)
    metric_label = 'all metrics' if aggregate_metrics or metric is None else metric
    ax.set_title(
        f"{y_label} | Disease ratio: {format_ratio(disease_ratio)} | Dataset: {dataset_type} | Metric: {metric_label}"
    )
    ax.set_xticks(x)
    ax.set_xticklabels([str(size) for size in sizes])
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=24)
    plt.tight_layout()

    out_dir = os.path.join(
        directory,
        f"{y_label}_PLOTS_MEAN_BY_SIZE",
        disease if disease else 'ALL',
    )
    if not aggregate_metrics and metric is not None:
        out_dir = os.path.join(out_dir, metric)
    os.makedirs(out_dir, exist_ok=True)

    suffix = 'all_metrics' if aggregate_metrics else f"metric_{metric}"
    fname = f"{y_label}_mean_vs_size_ratio_{ratio_value}_{suffix}_{dataset_type}.png"
    plt.savefig(os.path.join(out_dir, fname), bbox_inches='tight')
    plt.close()


In [ ]:
def rank_methods_per_row(pivot_df):
    """Return per-row ranks (1 = best) for numeric method columns."""
    method_cols = pivot_df.select_dtypes(include='number').columns
    return (
        pivot_df[method_cols]
        .rank(axis=1, method='min', ascending=True)
        .astype(int)
    )


In [ ]:
df_long = wide_to_long(std_mae_compilation_train_all)
df_long = add_nb_patients_and_diseased(df_long)
df_long = df_long[df_long['robust_method'].isin(ROBUST_METHODS)]

sites_with_nan_std = (
    df_long
    .groupby('site')
    .filter(lambda group: group.isna().any().any())
    ['site']
    .unique()
)

n_nan_sites_std = len(sites_with_nan_std)
n_total_sites_std = df_long['site'].nunique()

print(
    f"Sites excluded because of NaN values: {n_nan_sites_std} / {n_total_sites_std}"
)
if n_nan_sites_std:
    print('Sites:', list(sites_with_nan_std))

df_long = df_long[~df_long['site'].isin(sites_with_nan_std)].copy()

pivot_df_std = df_long.pivot_table(
    index=[
        'site',
        'disease',
        'metric',
        'bundle',
        'num_patients',
        'disease_ratio',
        'num_diseased',
    ],
    columns='robust_method',
    values='mae',
    aggfunc='first',
)


In [ ]:
rename_map = {}
if 'NO' in pivot_df_std.columns:
    rename_map['NO'] = 'NO_FILTERING'
if 'MLP_EXAMPLE' in pivot_df_std.columns:
    rename_map['MLP_EXAMPLE'] = 'MLP'

if rename_map:
    pivot_df_std = pivot_df_std.rename(columns=rename_map)


## Plot execution


In [ ]:
pivot_df_std = pivot_df_std.drop(columns=['raw', 'FLIP'], errors='ignore')

for disease_ratio in [50, 70, 80]:
    plot_mean_vs_sample_size_fixed_ratio(
        pivot_df_std,
        disease_ratio=disease_ratio,
        directory=MAE_PLOT_FOLDER,
        dataset_type='train',
        y_label='STD_MAE',
        disease='ALL',
        aggregate_metrics=True,
    )


In [ ]:
tasks = [
    (pivot_df_std, sample_size, MAE_PLOT_FOLDER, 'train', 'STD_MAE')
    for sample_size in sample_sizes
]

# Mean across diseases and metrics
Parallel(n_jobs=N_JOBS)(
    delayed(plot_mae_mean_all_diseases_metrics)(*task) for task in tasks
)

# Mean across diseases, metrics, and ratios
Parallel(n_jobs=N_JOBS)(
    delayed(plot_mae_mean_all_ratios)(*task) for task in tasks
)

tasks = [
    (pivot_df_std, sample_size, disease, MAE_PLOT_FOLDER, 'train', 'STD_MAE')
    for disease in diseases
    for sample_size in sample_sizes
]

# Mean across metrics per disease
Parallel(n_jobs=N_JOBS)(
    delayed(plot_rank_all_metrics)(*task) for task in tasks
)

tasks = [
    (pivot_df_std, sample_size, disease, metric, MAE_PLOT_FOLDER, 'train', 'STD_MAE')
    for disease in diseases
    for sample_size in sample_sizes
    for metric in metrics
]

# Mean per disease per metric across bundles
Parallel(n_jobs=N_JOBS)(
    delayed(plot_rank)(*task) for task in tasks
)

# Boxplots per disease per metric per bundle
Parallel(n_jobs=N_JOBS)(
    delayed(plot_mae_each_bundle)(*task) for task in tasks
)
